In [ ]:
!pip install opencv-python
import os
from PIL import Image, ImageDraw
import numpy as np
import cv2

from google.colab import drive
try:
    drive.flush_and_unmount()
except Exception:
    pass
drive.mount('/content/drive', force_remount=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# BASIC CIRCULAR CROP

In [ ]:
def process_folder(input_folder: str, output_folder: str):
    if not os.path.isdir(input_folder):
        print(f"Folder input tidak ditemukan: {input_folder}")
        print("Pastikan path-nya sudah benar (cek lewat panel Files di sidebar Colab).")
        return

    os.makedirs(output_folder, exist_ok=True)

    files = [
        f for f in os.listdir(input_folder)
        if f.lower().endswith(VALID_EXTENSIONS)
    ]

    if not files:
        print(f"Tidak ada gambar ditemukan di: {input_folder}")
        return

    print(f"Ditemukan {len(files)} gambar. Memproses...\n")

    for filename in files:
        input_path = os.path.join(input_folder, filename)
        name_only, _ext = os.path.splitext(filename)

        output_filename_check = name_only + ".png"
        output_path_check = os.path.join(output_folder, output_filename_check)
        if os.path.exists(output_path_check):
            print(f"  SKIP : {filename}  (sudah pernah diproses)")
            continue

        try:
            with Image.open(input_path) as img:
                original_size = img.size
                cropped = circular_crop(img, zoom=ZOOM)

                assert cropped.size == original_size, "Ukuran berubah!"

                output_filename = name_only + ".png"
                output_path = os.path.join(output_folder, output_filename)

                cropped.save(output_path, "PNG")
                print(f"  OK  : {filename}  ->  {output_filename}  "
                      f"(ukuran: {original_size[0]}x{original_size[1]})")

        except OSError as e:
            if e.errno == 107:
                # Mount Drive terputus -> file lain di belakangnya juga
                # akan gagal semua. Hentikan proses biar tidak spam GAGAL.
                print(f"  GAGAL: {filename}  ->  Koneksi Google Drive terputus ({e})")
                print("\n>> Mount Drive kena disconnect. Jalankan ulang CELL 1,")
                print(">> lalu jalankan CELL 2 ini lagi (file yang sudah OK tidak perlu diulang).")
                break
            print(f"  GAGAL: {filename}  ->  {e}")
        except Exception as e:
            print(f"  GAGAL: {filename}  ->  {e}")

    print(f"\nSelesai. Hasil tersimpan di: {output_folder}")

# SKENARIO 1

In [ ]:
# ZOOM: dipakai HANYA kalau AUTO_TRIM = False (lihat di bawah).
# 1.0  = tidak zoom, lingkaran mepet ke sisi terpendek foto
# 1.3+ = zoom lebih dekat ke tengah
ZOOM = 1.3

# AUTO_TRIM: kalau True, script otomatis MENCARI posisi objek/subjek di
# dalam foto (dengan mendeteksi area yang BUKAN background gelap/hitam),
# lalu lingkaran dipasang pas mengikuti objek itu -- bukan cuma zoom ke
# tengah frame. Ini yang menyelesaikan masalah "background hitam masih
# ikut kepotong" kalau objeknya kecil / tidak di tengah persis.
AUTO_TRIM = True

# Seberapa gelap dianggap "background" (0-255). Pixel dengan brightness
# di bawah angka ini dianggap background dan diabaikan saat mencari
# posisi objek. Naikkan kalau background bukan hitam pekat (misal abu2 gelap).
BLACK_THRESHOLD = 20

# Padding tambahan di sekitar objek yang terdeteksi (1.0 = pas-pasan,
# 1.15 = ada sedikit ruang napas di sekitar objek). Naikkan kalau hasil
# crop terlalu ketat/mepet ke objek.
TRIM_PADDING = 1.0

# DEBUG: kalau True, script juga menyimpan versi foto dengan kotak merah
# di sekitar area objek yang terdeteksi, ke folder DEBUG_FOLDER di bawah.
# Berguna untuk mengecek apakah deteksi objeknya benar atau meleset.
DEBUG = False
#DEBUG_FOLDER = OUTPUT_FOLDER + "_debug"

# MIN_CONTENT_AREA_RATIO: batas minimum ukuran objek supaya dianggap
# "objek asli" (bukan noise). Nilainya rasio terhadap luas foto (0-1).
# Contoh: 0.01 = objek harus minimal 1% dari luas foto untuk terdeteksi.
# Kalau objek di foto kamu ada yang KECIL (misal cuma beberapa persen
# dari luas foto) dan ternyata tidak ke-detect, TURUNKAN angka ini,
# misal ke 0.002 (0.2%) atau lebih kecil lagi.
MIN_CONTENT_AREA_RATIO = 0.002

VALID_EXTENSIONS = (".png", ".jpg", ".jpeg", ".bmp", ".webp", ".tiff", ".tif")


def get_content_bbox(image: Image.Image, threshold: int = 20):
    """
    Mencari bounding box objek utama di foto memakai deteksi contour
    (bukan cuma threshold biasa). Ini jauh lebih tahan terhadap noise
    dibanding metode getbbox() sederhana -- karena hanya BLOB TERBESAR
    yang dianggap objek, noise/artefak kecil di background diabaikan.

    Return (left, top, right, bottom) atau None kalau tidak ada objek
    yang cukup jelas terdeteksi.
    """
    arr = np.array(image.convert("L"))

    # Kurangi noise dulu sebelum threshold
    arr = cv2.GaussianBlur(arr, (5, 5), 0)

    _, bw = cv2.threshold(arr, threshold, 255, cv2.THRESH_BINARY)

    # Morphological opening: menghapus titik-titik noise kecil yang
    # ikut kedeteksi terang, tanpa menghapus objek utama yang besar
    kernel = np.ones((7, 7), np.uint8)
    bw = cv2.morphologyEx(bw, cv2.MORPH_OPEN, kernel, iterations=2)

    contours, _ = cv2.findContours(bw, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None

    # Ambil HANYA contour (blob) TERBESAR -> ini objek utamanya
    largest = max(contours, key=cv2.contourArea)

    # Kalau blob terbesar terlalu kecil (cuma noise), anggap tidak ada objek
    img_area = arr.shape[0] * arr.shape[1]
    if cv2.contourArea(largest) < img_area * MIN_CONTENT_AREA_RATIO:
        return None

    x, y, w, h = cv2.boundingRect(largest)
    return (x, y, x + w, y + h)


def circular_crop(image: Image.Image, zoom: float = 1.0,
                   auto_trim: bool = True, threshold: int = 20,
                   padding: float = 1.15) -> Image.Image:
    """
    Membuat crop lingkaran pada gambar, ukuran canvas OUTPUT tetap sama
    dengan aslinya (width x height tidak berubah).

    Kalau auto_trim=True: script mendeteksi dulu posisi objek/subjek di
    foto (area yang bukan background gelap), lalu lingkaran dipasang
    mengikuti posisi & ukuran objek itu (dengan sedikit padding).

    Kalau auto_trim=False atau objek tidak terdeteksi: fallback ke mode
    zoom manual dari tengah frame seperti sebelumnya.
    """
    image = image.convert("RGBA")
    width, height = image.size
    diameter = min(width, height)

    content_box = None
    if auto_trim:
        content_box = get_content_bbox(image, threshold=threshold)

    if content_box is not None:
        cl, ct, cr, cb = content_box
        content_w = cr - cl
        content_h = cb - ct
        content_cx = cl + content_w / 2
        content_cy = ct + content_h / 2

        # Ukuran crop = sisi terpanjang objek, dikasih padding
        crop_size = int(max(content_w, content_h) * padding)
        crop_size = max(1, min(crop_size, width, height))  # jangan lebih besar dari foto

        left = int(content_cx - crop_size / 2)
        top = int(content_cy - crop_size / 2)

        # Pastikan area crop tidak keluar dari batas foto
        left = max(0, min(left, width - crop_size))
        top = max(0, min(top, height - crop_size))
        right = left + crop_size
        bottom = top + crop_size
    else:
        # Fallback: mode zoom manual dari tengah frame
        crop_size = max(1, int(diameter / zoom))
        left = (width - crop_size) // 2
        top = (height - crop_size) // 2
        right = left + crop_size
        bottom = top + crop_size

    # Ambil area yang sudah ditentukan, lalu resize supaya pas mengisi diameter lingkaran
    center_crop = image.crop((left, top, right, bottom))
    center_crop = center_crop.resize((diameter, diameter), Image.LANCZOS)

    # Buat mask lingkaran seukuran diameter
    mask = Image.new("L", (diameter, diameter), 0)
    draw = ImageDraw.Draw(mask)
    draw.ellipse((0, 0, diameter, diameter), fill=255)

    circle_img = Image.new("RGBA", (diameter, diameter), (0, 0, 0, 0))
    circle_img.paste(center_crop, (0, 0), mask=mask)

    # Tempel lingkaran ke canvas seukuran foto ASLI (ukuran tidak berubah).
    # PENTING: posisi tempelnya harus PAS di tengah OBJEK yang terdeteksi
    # (left/top di atas), BUKAN di tengah kanvas -- supaya kalau objeknya
    # tadinya tidak center di foto, hasil lingkarannya tetap sesuai posisi
    # objek yang sebenarnya, bukan meloncat ke tengah frame.
    result = Image.new("RGBA", (width, height), (0, 0, 0, 0))

    # Titik tengah area crop asli (left/top/right/bottom di atas)
    crop_center_x = left + crop_size / 2
    crop_center_y = top + crop_size / 2

    paste_x = int(round(crop_center_x - diameter / 2))
    paste_y = int(round(crop_center_y - diameter / 2))

    # Pastikan lingkaran tidak keluar dari batas kanvas
    paste_x = max(0, min(paste_x, width - diameter))
    paste_y = max(0, min(paste_y, height - diameter))

    result.paste(circle_img, (paste_x, paste_y), mask=circle_img)

    return result


def process_folder(input_folder: str, output_folder: str):
    if not os.path.isdir(input_folder):
        print(f"Folder input tidak ditemukan: {input_folder}")
        print("Pastikan path-nya sudah benar (cek lewat panel Files di sidebar Colab).")
        return

    os.makedirs(output_folder, exist_ok=True)

    files = [
        f for f in os.listdir(input_folder)
        if f.lower().endswith(VALID_EXTENSIONS)
    ]

    if not files:
        print(f"Tidak ada gambar ditemukan di: {input_folder}")
        return

    if DEBUG:
        os.makedirs(DEBUG_FOLDER, exist_ok=True)

    print(f"Ditemukan {len(files)} gambar. Memproses...\n")

    for filename in files:
        input_path = os.path.join(input_folder, filename)
        name_only, _ext = os.path.splitext(filename)

        output_filename_check = name_only + ".png"
        output_path_check = os.path.join(output_folder, output_filename_check)
        if os.path.exists(output_path_check):
            print(f"  SKIP : {filename}  (sudah pernah diproses)")
            continue

        try:
            with Image.open(input_path) as img:
                original_size = img.size

                if DEBUG:
                    box = get_content_bbox(img.convert("RGBA"), threshold=BLACK_THRESHOLD)
                    debug_img = img.convert("RGB").copy()
                    if box is not None:
                        d = ImageDraw.Draw(debug_img)
                        d.rectangle(box, outline=(255, 0, 0), width=5)
                    debug_img.save(os.path.join(DEBUG_FOLDER, name_only + "_debug.jpg"))

                cropped = circular_crop(
                    img,
                    zoom=ZOOM,
                    auto_trim=AUTO_TRIM,
                    threshold=BLACK_THRESHOLD,
                    padding=TRIM_PADDING,
                )

                assert cropped.size == original_size, "Ukuran berubah!"

                output_filename = name_only + ".png"
                output_path = os.path.join(output_folder, output_filename)

                cropped.save(output_path, "PNG")
                print(f"  OK  : {filename}  ->  {output_filename}  "
                      f"(ukuran: {original_size[0]}x{original_size[1]})")

        except OSError as e:
            if e.errno == 107:
                # Mount Drive terputus -> file lain di belakangnya juga
                # akan gagal semua. Hentikan proses biar tidak spam GAGAL.
                print(f"  GAGAL: {filename}  ->  Koneksi Google Drive terputus ({e})")
                print("\n>> Mount Drive kena disconnect. Jalankan ulang CELL 1,")
                print(">> lalu jalankan CELL ini lagi (file yang sudah OK tidak perlu diulang).")
                break
            print(f"  GAGAL: {filename}  ->  {e}")
        except Exception as e:
            print(f"  GAGAL: {filename}  ->  {e}")

    print(f"\nSelesai. Hasil tersimpan di: {output_folder}")

#SKENARIO 2

In [ ]:
# THRESHOLD: batas kecerahan (grayscale) yang dipakai buat memisahkan
# objek dari background gelap/hitam. Pixel dengan nilai grayscale
# DI ATAS threshold ini dianggap "isi" (bagian dari objek), dan yang
# di bawahnya dianggap background lalu dibuang saat cropImageFromGray.
THRESHOLD = 7

# SIGMA: sigma untuk Gaussian blur yang dipakai di proses "blending"
# (langkah addWeighted). Makin besar sigma, makin halus/lebar blur-nya.
SIGMA = 10

# GAMMA: bobot statis (skalar) yang ditambahkan ke semua pixel hasil
# akhir saat proses blending (addWeighted).
GAMMA = 128

# ALPHA: bobot untuk gambar hasil masking lingkaran saat blending.
ALPHA = 4

# BETA: bobot untuk gambar hasil Gaussian blur saat blending.
# Biasanya negatif supaya blur-nya "mengurangi" pixel asli (efek
# semacam unsharp masking / penekanan highlight).
BETA = -4

VALID_EXTENSIONS = (".png", ".jpg", ".jpeg", ".bmp", ".webp", ".tiff", ".tif")


# ============================================================
# CELL 3 — ALGORITHM 1: Crop Image From Grey Scale
# ============================================================
def crop_image_from_gray(image: np.ndarray, threshold: int = THRESHOLD) -> np.ndarray:
    """
    Memotong (crop) gambar mengikuti bounding box area yang "terang"
    (di atas threshold), membuang baris/kolom yang isinya cuma
    background gelap/hitam.

    - Kalau image grayscale (ndim == 2): langsung mask & crop.
    - Kalau image berwarna: mask dihitung dari versi grayscale-nya,
      lalu diterapkan ke tiap channel (R, G, B) secara terpisah,
      baru digabung lagi jadi satu gambar berwarna.
    """
    if image.ndim == 2:
        mask = image > threshold
        if not mask.any():
            return image
        segmented_image = image[np.ix_(mask.any(axis=1), mask.any(axis=0))]
        return segmented_image
    else:
        gray_image = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        mask = gray_image > threshold

        if not mask.any():
            # Seluruh gambar di bawah threshold -> tidak ada yang bisa
            # di-crop, kembalikan gambar aslinya supaya tidak error.
            return image

        rows = mask.any(axis=1)
        cols = mask.any(axis=0)

        red_channel = image[:, :, 0][np.ix_(rows, cols)]
        green_channel = image[:, :, 1][np.ix_(rows, cols)]
        blue_channel = image[:, :, 2][np.ix_(rows, cols)]

        segmented_image = np.stack([red_channel, green_channel, blue_channel], axis=-1)
        return segmented_image


# ============================================================
# CELL 4 — ALGORITHM 2: crop -> mask lingkaran -> crop lagi -> blend blur
# ============================================================
def process_image(image: np.ndarray, sigma: float = SIGMA, gamma: float = GAMMA,
                   alpha: float = ALPHA, beta: float = BETA) -> np.ndarray:
    """
    1. Crop gambar dari background gelap (cropImageFromGray)
    2. Buat mask lingkaran di tengah, radius = sisi terpendek / 2
    3. Terapkan mask lingkaran ke gambar (bitwise_and)
    4. Crop ulang hasilnya (cropImageFromGray lagi)
    5. Hitung Gaussian blur
    6. Gabungkan (blend) hasil masking dengan blur pakai addWeighted
    """
    # Langkah 2: crop pertama
    img = crop_image_from_gray(image, threshold=THRESHOLD)

    height, width = img.shape[:2]
    x = width // 2
    y = height // 2
    radius = min(x, y)

    # Langkah 6-8: mask lingkaran hitam-putih seukuran img
    circle_image = np.zeros((height, width), dtype=np.uint8)
    cv2.circle(circle_image, (x, y), radius, 255, thickness=-1)

    # Langkah 9: terapkan mask ke gambar (area di luar lingkaran jadi hitam)
    image_processed = cv2.bitwise_and(img, img, mask=circle_image)

    # Langkah 10: crop ulang, biar sisa border hitam dari masking
    # lingkaran ikut kebuang juga
    image_processed = crop_image_from_gray(image_processed, threshold=THRESHOLD)

    # Langkah 11: Gaussian blur
    blur = cv2.GaussianBlur(img, (0, 0), sigma)

    # Crop kedua bisa membuat ukuran image_processed berbeda dari
    # img/blur, jadi disamakan dulu sebelum di-blend supaya
    # addWeighted tidak error karena shape mismatch.
    if blur.shape != image_processed.shape:
        blur = cv2.resize(blur, (image_processed.shape[1], image_processed.shape[0]))

    # Langkah 12: blend
    image_processed = cv2.addWeighted(image_processed, alpha, blur, beta, gamma)

    return image_processed

def process_folder(input_folder: str, output_folder: str):
    if not os.path.isdir(input_folder):
        print(f"Folder input tidak ditemukan: {input_folder}")
        print("Pastikan path-nya sudah benar (cek lewat panel Files di sidebar Colab,")
        print("klik kanan folder di Google Drive > Copy path).")
        return

    os.makedirs(output_folder, exist_ok=True)

    files = [
        f for f in os.listdir(input_folder)
        if f.lower().endswith(VALID_EXTENSIONS)
    ]

    if not files:
        print(f"Tidak ada gambar ditemukan di: {input_folder}")
        return

    print(f"Ditemukan {len(files)} gambar. Memproses...\n")

    for filename in files:
        input_path = os.path.join(input_folder, filename)
        name_only, _ext = os.path.splitext(filename)

        output_filename = name_only + ".png"
        output_path = os.path.join(output_folder, output_filename)
        if os.path.exists(output_path):
            print(f"  SKIP : {filename}  (sudah pernah diproses)")
            continue

        try:
            # Baca lewat PIL dulu supaya format warna konsisten
            # (termasuk gambar dengan alpha channel / mode aneh),
            # lalu convert ke array RGB buat diproses cv2.
            with Image.open(input_path) as pil_img:
                rgb_img = pil_img.convert("RGB")
                arr = np.array(rgb_img)  # urutan channel: RGB

            original_size = (arr.shape[1], arr.shape[0])

            processed = process_image(arr)

            # Simpan hasil. Array masih dalam urutan channel RGB dari
            # PIL, jadi convert balik ke BGR sebelum ditulis via cv2.
            out_bgr = cv2.cvtColor(processed, cv2.COLOR_RGB2BGR)
            cv2.imwrite(output_path, out_bgr)

            print(f"  OK  : {filename}  ->  {output_filename}  "
                  f"(ukuran asli: {original_size[0]}x{original_size[1]}, "
                  f"ukuran hasil: {processed.shape[1]}x{processed.shape[0]})")

        except OSError as e:
            if e.errno == 107:
                # Mount Drive terputus -> file lain di belakangnya juga
                # akan gagal semua. Hentikan proses biar tidak spam GAGAL.
                print(f"  GAGAL: {filename}  ->  Koneksi Google Drive terputus ({e})")
                print("\n>> Mount Drive kena disconnect. Jalankan ulang CELL 1,")
                print(">> lalu jalankan CELL ini lagi (file yang sudah OK tidak perlu diulang).")
                break
            print(f"  GAGAL: {filename}  ->  {e}")
        except Exception as e:
            print(f"  GAGAL: {filename}  ->  {e}")

    print(f"\nSelesai. Hasil tersimpan di: {output_folder}")

#BOUNDING BOX CROP


In [ ]:
!pip install opencv-python
import os
import numpy as np
import cv2
from PIL import Image, ImageDraw
from tqdm.auto import tqdm

# ZOOM: dipakai HANYA kalau AUTO_TRIM = False (lihat di bawah).
# 1.0  = tidak zoom, crop mepet ke sisi terpendek foto
# 1.3+ = zoom lebih dekat ke tengah
ZOOM = 1.3

# AUTO_TRIM: kalau True, script otomatis MENCARI posisi objek/subjek di
# dalam foto (dengan mendeteksi area yang BUKAN background gelap/hitam),
# lalu foto DIPOTONG (crop) pas mengikuti bounding box objek itu -- bukan
# cuma zoom ke tengah frame. Ini yang menyelesaikan masalah "background
# hitam masih ikut kepotong" kalau objeknya kecil / tidak di tengah persis.
AUTO_TRIM = True

# Seberapa gelap dianggap "background" (0-255). Pixel dengan brightness
# di bawah angka ini dianggap background dan diabaikan saat mencari
# posisi objek. Naikkan kalau background bukan hitam pekat (misal abu2 gelap).
BLACK_THRESHOLD = 20

# Padding tambahan di sekitar objek yang terdeteksi (1.0 = pas-pasan,
# 1.15 = ada sedikit ruang napas di sekitar objek). Naikkan kalau hasil
# crop terlalu ketat/mepet ke objek.
TRIM_PADDING = 1.0

# DEBUG: kalau True, script juga menyimpan versi foto dengan kotak merah
# di sekitar area objek yang terdeteksi, ke folder DEBUG_FOLDER di bawah.
# Berguna untuk mengecek apakah deteksi objeknya benar atau meleset.
DEBUG = False
#DEBUG_FOLDER = OUTPUT_FOLDER + "_debug"

# MIN_CONTENT_AREA_RATIO: batas minimum ukuran objek supaya dianggap
# "objek asli" (bukan noise). Nilainya rasio terhadap luas foto (0-1).
# Contoh: 0.01 = objek harus minimal 1% dari luas foto untuk terdeteksi.
# Kalau objek di foto kamu ada yang KECIL (misal cuma beberapa persen
# dari luas foto) dan ternyata tidak ke-detect, TURUNKAN angka ini,
# misal ke 0.002 (0.2%) atau lebih kecil lagi.
MIN_CONTENT_AREA_RATIO = 0.002

VALID_EXTENSIONS = (".png", ".jpg", ".jpeg", ".bmp", ".webp", ".tiff", ".tif")


def get_content_bbox(image: Image.Image, threshold: int = 20):
    """
    Mencari bounding box objek utama di foto memakai deteksi contour
    (bukan cuma threshold biasa). Ini jauh lebih tahan terhadap noise
    dibanding metode getbbox() sederhana -- karena hanya BLOB TERBESAR
    yang dianggap objek, noise/artefak kecil di background diabaikan.

    Return (left, top, right, bottom) atau None kalau tidak ada objek
    yang cukup jelas terdeteksi.
    """
    arr = np.array(image.convert("L"))

    # Kurangi noise dulu sebelum threshold
    arr = cv2.GaussianBlur(arr, (5, 5), 0)

    _, bw = cv2.threshold(arr, threshold, 255, cv2.THRESH_BINARY)

    # Morphological opening: menghapus titik-titik noise kecil yang
    # ikut kedeteksi terang, tanpa menghapus objek utama yang besar
    kernel = np.ones((7, 7), np.uint8)
    bw = cv2.morphologyEx(bw, cv2.MORPH_OPEN, kernel, iterations=2)

    contours, _ = cv2.findContours(bw, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None

    # Ambil HANYA contour (blob) TERBESAR -> ini objek utamanya
    largest = max(contours, key=cv2.contourArea)

    # Kalau blob terbesar terlalu kecil (cuma noise), anggap tidak ada objek
    img_area = arr.shape[0] * arr.shape[1]
    if cv2.contourArea(largest) < img_area * MIN_CONTENT_AREA_RATIO:
        return None

    x, y, w, h = cv2.boundingRect(largest)
    return (x, y, x + w, y + h)


def bbox_crop(image: Image.Image, zoom: float = 1.0,
              auto_trim: bool = True, threshold: int = 20,
              padding: float = 1.15) -> Image.Image:
    """
    Memotong (crop) gambar sesuai bounding box objek -- TANPA lingkaran,
    TANPA resize, TANPA dipasang balik ke canvas ukuran asli. Hasil
    akhirnya adalah potongan foto asli (rectangular), ukurannya sesuai
    area crop yang ditentukan (biasanya lebih kecil dari foto asli).

    Kalau auto_trim=True: script mendeteksi dulu posisi objek/subjek di
    foto (area yang bukan background gelap), lalu foto dipotong mengikuti
    bounding box objek itu (dengan sedikit padding), berbentuk kotak
    (persegi) yang sisinya = sisi terpanjang objek x padding.

    Kalau auto_trim=False atau objek tidak terdeteksi: fallback ke mode
    zoom manual dari tengah frame (crop persegi berukuran
    sisi_terpendek_foto / zoom, diambil dari tengah).
    """
    image = image.convert("RGBA")
    width, height = image.size

    content_box = None
    if auto_trim:
        content_box = get_content_bbox(image, threshold=threshold)

    if content_box is not None:
        cl, ct, cr, cb = content_box
        content_w = cr - cl
        content_h = cb - ct
        content_cx = cl + content_w / 2
        content_cy = ct + content_h / 2

        # Ukuran crop = sisi terpanjang objek, dikasih padding
        crop_size = int(max(content_w, content_h) * padding)
        crop_size = max(1, min(crop_size, width, height))  # jangan lebih besar dari foto

        left = int(content_cx - crop_size / 2)
        top = int(content_cy - crop_size / 2)

        # Pastikan area crop tidak keluar dari batas foto
        left = max(0, min(left, width - crop_size))
        top = max(0, min(top, height - crop_size))
        right = left + crop_size
        bottom = top + crop_size
    else:
        # Fallback: mode zoom manual dari tengah frame
        diameter = min(width, height)
        crop_size = max(1, int(diameter / zoom))
        left = (width - crop_size) // 2
        top = (height - crop_size) // 2
        right = left + crop_size
        bottom = top + crop_size

    # Cukup potong (crop) area yang sudah ditentukan -- tidak ada resize,
    # tidak ada mask lingkaran, tidak ditempel balik ke canvas asli.
    result = image.crop((left, top, right, bottom))

    return result


def process_folder(input_folder: str, output_folder: str):
    if not os.path.isdir(input_folder):
        print(f"Folder input tidak ditemukan: {input_folder}")
        print("Pastikan path-nya sudah benar (cek lewat panel Files di sidebar Colab).")
        return

    os.makedirs(output_folder, exist_ok=True)

    files = [
        f for f in os.listdir(input_folder)
        if f.lower().endswith(VALID_EXTENSIONS)
    ]

    if not files:
        print(f"Tidak ada gambar ditemukan di: {input_folder}")
        return

    if DEBUG:
        os.makedirs(DEBUG_FOLDER, exist_ok=True)

    print(f"Ditemukan {len(files)} gambar. Memproses...\n")

    failed_files = []       # (filename, alasan) yang gagal diproses
    drive_disconnected = False

    progress = tqdm(files, desc="Memproses gambar", unit="gambar")

    for filename in progress:
        input_path = os.path.join(input_folder, filename)
        name_only, _ext = os.path.splitext(filename)

        output_filename_check = name_only + ".png"
        output_path_check = os.path.join(output_folder, output_filename_check)
        if os.path.exists(output_path_check):
            continue  # sudah pernah diproses, lewati diam-diam

        try:
            with Image.open(input_path) as img:
                if DEBUG:
                    box = get_content_bbox(img.convert("RGBA"), threshold=BLACK_THRESHOLD)
                    debug_img = img.convert("RGB").copy()
                    if box is not None:
                        d = ImageDraw.Draw(debug_img)
                        d.rectangle(box, outline=(255, 0, 0), width=5)
                    debug_img.save(os.path.join(DEBUG_FOLDER, name_only + "_debug.jpg"))

                cropped = bbox_crop(
                    img,
                    zoom=ZOOM,
                    auto_trim=AUTO_TRIM,
                    threshold=BLACK_THRESHOLD,
                    padding=TRIM_PADDING,
                )

                output_filename = name_only + ".png"
                output_path = os.path.join(output_folder, output_filename)

                cropped.save(output_path, "PNG")

        except OSError as e:
            if e.errno == 107:
                # Mount Drive terputus -> file lain di belakangnya juga
                # akan gagal semua. Hentikan proses biar tidak spam GAGAL.
                failed_files.append((filename, "Koneksi Google Drive terputus"))
                drive_disconnected = True
                break
            failed_files.append((filename, str(e)))
        except Exception as e:
            failed_files.append((filename, str(e)))

    progress.close()

    print(f"\nSelesai. Hasil tersimpan di: {output_folder}")

    if failed_files:
        print(f"\n{len(failed_files)} gambar GAGAL diproses:")
        for filename, reason in failed_files:
            print(f"  - {filename}  ({reason})")
    else:
        print("\nSemua gambar berhasil diproses, tidak ada yang gagal.")

    if drive_disconnected:
        print("\n>> Mount Drive kena disconnect. Jalankan ulang CELL 1,")
        print(">> lalu jalankan CELL ini lagi (file yang sudah OK tidak perlu diulang).")

# DEBUG

In [ ]:
!pip install opencv-python
import os
import numpy as np
import cv2
from PIL import Image, ImageDraw
from tqdm.auto import tqdm

# ZOOM: dipakai HANYA kalau tidak ada objek terdeteksi sama sekali
# (fallback mode). 1.0 = tidak zoom, crop mepet ke sisi terpendek foto.
ZOOM = 1.3

# CROP_MODE: pilih strategi crop.
#
#   "bbox"      (DEFAULT, direkomendasikan untuk kebanyakan kasus)
#               Crop mengikuti bounding box objek APA ADANYA -- hasilnya
#               tetap terlihat "utuh bulat matanya", cuma buang kelebihan
#               bingkai hitam di sekelilingnya. Kalau ada notch/goresan
#               kecil di pinggir (wajar di foto fundus), itu TETAP ikut
#               kepotong ke dalam hasil -- ini normal dan tidak masalah.
#
#   "inscribed" Cari persegi panjang terbesar yang 100% BEBAS dari
#               background/notch sama sekali. Hasilnya DIJAMIN bersih,
#               TAPI otomatis lebih kecil dari lingkaran aslinya (untuk
#               objek bulat, secara geometri paling besar cuma ~70% dari
#               diameter, walau tidak ada notch sama sekali). Pakai mode
#               ini HANYA kalau fotomu punya notch/artefak besar yang
#               harus dibuang total, dan kamu tidak keberatan kehilangan
#               cukup banyak bagian tepi retina yang sebenarnya sehat.
CROP_MODE = "bbox"

AUTO_TRIM = True

# Seberapa gelap dianggap "background" (0-255).
BLACK_THRESHOLD = 20

# FORCE_SQUARE: kalau True, hasil crop dipaksa jadi persegi (sisi =
# sisi terpanjang [mode bbox] atau terpendek [mode inscribed]) supaya
# semua output punya rasio 1:1. Kalau False (default), crop mengikuti
# bentuk aslinya apa adanya.
FORCE_SQUARE = False

# Padding di sekitar bounding box objek (mode "bbox"): 1.0 = pas-pasan
# ke tepi objek, 1.05 = kasih 5% ruang napas di semua sisi.
TRIM_PADDING = 1.02

# SAFETY_MARGIN (mode "inscribed"): 1.0 = pakai persegi panjang terbesar
# apa adanya. Turunkan sedikit (misal 0.97) untuk margin aman tambahan.
# JANGAN diisi > 1.0.
SAFETY_MARGIN = 1.0

# Resolusi kerja saat mode "inscribed" mencari persegi panjang terbesar
# (downsample dulu supaya cepat, koordinat di-scale balik ke resolusi asli).
SEARCH_MAX_DIM = 500

# DEBUG: kalau True, simpan versi foto dengan kotak merah di sekitar area
# crop yang terdeteksi, ke folder DEBUG_FOLDER, untuk mengecek hasil deteksi.
DEBUG = False
DEBUG_FOLDER = "debug_crop"

# MIN_CONTENT_AREA_RATIO: batas minimum ukuran objek supaya dianggap
# "objek asli" (bukan noise), sebagai rasio terhadap luas foto.
MIN_CONTENT_AREA_RATIO = 0.002

VALID_EXTENSIONS = (".png", ".jpg", ".jpeg", ".bmp", ".webp", ".tiff", ".tif")


def get_content_mask(image: Image.Image, threshold: int = 20):
    """
    Membuat mask biner (0/255) berisi HANYA blob objek terbesar (noise
    kecil di background sudah diabaikan). Return mask atau None kalau
    tidak ada objek yang cukup jelas terdeteksi.
    """
    arr = np.array(image.convert("L"))
    arr = cv2.GaussianBlur(arr, (5, 5), 0)

    _, bw = cv2.threshold(arr, threshold, 255, cv2.THRESH_BINARY)

    kernel = np.ones((7, 7), np.uint8)
    bw = cv2.morphologyEx(bw, cv2.MORPH_OPEN, kernel, iterations=2)

    contours, _ = cv2.findContours(bw, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None

    largest = max(contours, key=cv2.contourArea)

    img_area = arr.shape[0] * arr.shape[1]
    if cv2.contourArea(largest) < img_area * MIN_CONTENT_AREA_RATIO:
        return None

    clean_mask = np.zeros_like(bw)
    cv2.drawContours(clean_mask, [largest], -1, 255, thickness=cv2.FILLED)
    return clean_mask


def get_content_bbox(image: Image.Image, threshold: int = 20):
    """
    Mencari bounding box objek di foto.

    PENTING: lebar (kolom kiri-kanan) dan tinggi (baris atas-bawah)
    dihitung INDEPENDEN lewat proyeksi mask, dan mask yang dipakai adalah
    GABUNGAN semua blob yang signifikan -- BUKAN cuma blob/contour
    terbesar. Ini untuk mengatasi kasus dimana objek foto sebenarnya
    terpecah jadi beberapa bagian setelah threshold+morphology (misal
    karena pembuluh darah gelap, noda gelap, atau gradasi pinggir yang
    membelah mask) -- kalau cuma pakai "contour terbesar", potongan objek
    yang lain bisa ke-skip dan bikin bounding box salah/terlalu sempit di
    salah satu sisi (biasanya lebar), padahal potongan itu tetap bagian
    dari objek yang sama dan seharusnya ikut dihitung.

    Return (left, top, right, bottom) atau None kalau tidak ada objek
    yang cukup jelas terdeteksi.
    """
    arr = np.array(image.convert("L"))
    arr = cv2.GaussianBlur(arr, (5, 5), 0)

    _, bw = cv2.threshold(arr, threshold, 255, cv2.THRESH_BINARY)

    # Kernel kecil + 1 iterasi saja -- cukup untuk buang noise titik-titik
    # kecil, tapi tidak terlalu agresif mengikis/memutus bagian objek yang
    # tipis (yang sebelumnya bisa memicu masalah "objek terpecah" di atas).
    kernel = np.ones((3, 3), np.uint8)
    bw = cv2.morphologyEx(bw, cv2.MORPH_OPEN, kernel, iterations=1)

    contours, _ = cv2.findContours(bw, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None

    img_area = arr.shape[0] * arr.shape[1]

    # Ambang noise: pakai fraksi yang jauh lebih kecil dari
    # MIN_CONTENT_AREA_RATIO untuk MEMBUANG blob, supaya potongan kecil
    # dari objek yang terpecah tetap ikut dihitung -- yang benar-benar
    # dibuang cuma noise yang jelas-jelas remeh.
    noise_area_threshold = img_area * (MIN_CONTENT_AREA_RATIO / 20)

    significant = [c for c in contours if cv2.contourArea(c) >= noise_area_threshold]
    if not significant:
        return None

    # Objek utama (buat cek MIN_CONTENT_AREA_RATIO) tetap dari blob
    # terbesar, tapi bounding box akhirnya dari GABUNGAN semua blob
    # signifikan (union), jadi lebar & tinggi masing-masing dihitung
    # dari total sebaran semua bagian objek, bukan cuma satu blob.
    largest_area = max(cv2.contourArea(c) for c in significant)
    if largest_area < img_area * MIN_CONTENT_AREA_RATIO:
        return None

    union_mask = np.zeros_like(bw)
    cv2.drawContours(union_mask, significant, -1, 255, thickness=cv2.FILLED)

    # Proyeksi independen: kolom yang punya mask -> tentukan left/right;
    # baris yang punya mask -> tentukan top/bottom, terpisah satu sama lain.
    cols_with_content = np.where(union_mask.any(axis=0))[0]
    rows_with_content = np.where(union_mask.any(axis=1))[0]
    if cols_with_content.size == 0 or rows_with_content.size == 0:
        return None

    left, right = int(cols_with_content[0]), int(cols_with_content[-1]) + 1
    top, bottom = int(rows_with_content[0]), int(rows_with_content[-1]) + 1
    return (left, top, right, bottom)


def _largest_rectangle_in_histogram(heights):
    """
    Klasik: cari persegi panjang terbesar di histogram, sekalian posisinya
    (bukan cuma luasnya). Return (area, left, right_exclusive, height).
    """
    stack = []  # index kolom
    best_area, best_left, best_right, best_h = 0, 0, 0, 0
    n = len(heights)
    for i in range(n + 1):
        cur_h = heights[i] if i < n else 0
        while stack and heights[stack[-1]] >= cur_h:
            h = heights[stack.pop()]
            left = stack[-1] + 1 if stack else 0
            width = i - left
            area = h * width
            if area > best_area:
                best_area, best_left, best_right, best_h = area, left, i, h
        stack.append(i)
    return best_area, best_left, best_right, best_h


def largest_inscribed_rect(mask: np.ndarray, search_max_dim: int = 500):
    """
    Ini bagian INTI: mencari PERSEGI PANJANG TERBESAR (bentuk & posisi
    bebas, tidak harus di tengah, tidak harus persegi) yang seluruh
    isinya masih di dalam mask objek -- dijamin TIDAK menyentuh
    background maupun bagian objek yang "notch"/terpotong sebagian
    (karena bagian itu otomatis bukan bagian dari mask objek yang solid).

    Bekerja untuk bentuk objek apapun: bulat, oval, atau tidak beraturan
    sekalipun (misal ada sudut yang kepotong saat foto diambil).

    Mask dikecilkan dulu (downsample) untuk kecepatan, memakai INTER_AREA
    + threshold ketat supaya downsampling tetap KONSERVATIF (satu sel
    dianggap "aman" hanya kalau blok aslinya benar-benar penuh objek).

    Return (left, top, right, bottom) dalam koordinat pixel ASLI (skala
    penuh), atau None kalau mask kosong.
    """
    h, w = mask.shape
    scale = min(1.0, search_max_dim / max(h, w))
    small_w = max(1, int(w * scale))
    small_h = max(1, int(h * scale))

    small_mask = cv2.resize(mask, (small_w, small_h), interpolation=cv2.INTER_AREA)
    # threshold ketat: sel dianggap "objek" hanya kalau blok aslinya nyaris 100% putih
    bin_mask = (small_mask >= 250).astype(np.uint8)

    if bin_mask.sum() == 0:
        return None

    heights = [0] * small_w
    best = (0, 0, 0, 0, 0)  # area, top, left, bottom(exclusive), right(exclusive)

    for r in range(small_h):
        row = bin_mask[r]
        for c in range(small_w):
            heights[c] = heights[c] + 1 if row[c] else 0

        area, left, right, rect_h = _largest_rectangle_in_histogram(heights)
        if area > best[0]:
            top = r - rect_h + 1
            best = (area, top, left, r + 1, right)

    area, top, left, bottom, right = best
    if area == 0:
        return None

    # Scale balik ke koordinat penuh, dibulatkan KE DALAM (konservatif)
    # supaya hasilnya tetap dijamin di dalam mask asli.
    scale_x = w / small_w
    scale_y = h / small_h

    orig_left = int(np.ceil(left * scale_x))
    orig_top = int(np.ceil(top * scale_y))
    orig_right = int(np.floor(right * scale_x))
    orig_bottom = int(np.floor(bottom * scale_y))

    orig_right = max(orig_right, orig_left + 1)
    orig_bottom = max(orig_bottom, orig_top + 1)

    return (orig_left, orig_top, orig_right, orig_bottom)


def bbox_crop(image: Image.Image, zoom: float = 1.0,
              auto_trim: bool = True, threshold: int = 20,
              crop_mode: str = "bbox",
              trim_padding: float = 1.02,
              safety_margin: float = 1.0,
              force_square: bool = False,
              search_max_dim: int = 500) -> Image.Image:
    """
    Crop foto ke area objek yang terdeteksi.

    crop_mode = "bbox" (default): pakai bounding box objek apa adanya
    (ditambah trim_padding sedikit) -- hasilnya tetap terlihat utuh
    seperti aslinya, cuma bingkai hitam berlebih di sekelilingnya yang
    dibuang.

    crop_mode = "inscribed": cari persegi panjang terbesar yang 100%
    bebas dari background/notch (lebih agresif, hasil lebih kecil tapi
    dijamin bersih total).

    Kalau auto_trim=False atau objek tidak terdeteksi: fallback ke mode
    zoom manual dari tengah frame.
    """
    image = image.convert("RGBA")
    width, height = image.size

    if not auto_trim:
        return _fallback_zoom_crop(image, zoom)

    if crop_mode == "inscribed":
        mask = get_content_mask(image, threshold=threshold)
        if mask is None:
            return _fallback_zoom_crop(image, zoom)

        rect = largest_inscribed_rect(mask, search_max_dim=search_max_dim)
        if rect is None:
            return _fallback_zoom_crop(image, zoom)

        left, top, right, bottom = rect
        rect_w = right - left
        rect_h = bottom - top
        cx = left + rect_w / 2
        cy = top + rect_h / 2

        if force_square:
            side = min(rect_w, rect_h) * safety_margin
            crop_w = crop_h = side
        else:
            crop_w = rect_w * safety_margin
            crop_h = rect_h * safety_margin

        crop_w = max(1, int(crop_w))
        crop_h = max(1, int(crop_h))

        crop_left = int(cx - crop_w / 2)
        crop_top = int(cy - crop_h / 2)
        return image.crop((crop_left, crop_top, crop_left + crop_w, crop_top + crop_h))

    # crop_mode == "bbox" (default)
    content_box = get_content_bbox(image, threshold=threshold)
    if content_box is None:
        return _fallback_zoom_crop(image, zoom)

    cl, ct, cr, cb = content_box
    content_w = cr - cl
    content_h = cb - ct
    cx = cl + content_w / 2
    cy = ct + content_h / 2

    if force_square:
        side = max(content_w, content_h) * trim_padding
        crop_w = crop_h = side
    else:
        crop_w = content_w * trim_padding
        crop_h = content_h * trim_padding

    # Batasi ke ukuran foto asli (kalau padding bikin lebih besar dari
    # foto, ya otomatis dipotong sesuai batas foto -- ini normal, bukan
    # tanda gagal, cuma berarti objeknya memang nyaris memenuhi seluruh
    # frame dan tidak ada lagi bingkai hitam berlebih untuk dibuang)
    crop_w = max(1, min(int(crop_w), width))
    crop_h = max(1, min(int(crop_h), height))

    left = int(cx - crop_w / 2)
    top = int(cy - crop_h / 2)
    left = max(0, min(left, width - crop_w))
    top = max(0, min(top, height - crop_h))

    return image.crop((left, top, left + crop_w, top + crop_h))


def _fallback_zoom_crop(image: Image.Image, zoom: float) -> Image.Image:
    width, height = image.size
    diameter = min(width, height)
    crop_size = max(1, int(diameter / zoom))
    left = (width - crop_size) // 2
    top = (height - crop_size) // 2
    return image.crop((left, top, left + crop_size, top + crop_size))


def process_folder(input_folder: str, output_folder: str):
    if not os.path.isdir(input_folder):
        print(f"Folder input tidak ditemukan: {input_folder}")
        print("Pastikan path-nya sudah benar (cek lewat panel Files di sidebar Colab).")
        return

    os.makedirs(output_folder, exist_ok=True)

    files = [
        f for f in os.listdir(input_folder)
        if f.lower().endswith(VALID_EXTENSIONS)
    ]

    if not files:
        print(f"Tidak ada gambar ditemukan di: {input_folder}")
        return

    if DEBUG:
        os.makedirs(DEBUG_FOLDER, exist_ok=True)

    print(f"Ditemukan {len(files)} gambar. Memproses...\n")

    failed_files = []
    drive_disconnected = False

    progress = tqdm(files, desc="Memproses gambar", unit="gambar")

    for filename in progress:
        input_path = os.path.join(input_folder, filename)
        name_only, _ext = os.path.splitext(filename)

        output_filename_check = name_only + ".png"
        output_path_check = os.path.join(output_folder, output_filename_check)
        if os.path.exists(output_path_check):
            continue

        try:
            with Image.open(input_path) as img:
                if DEBUG:
                    debug_img = img.convert("RGB").copy()
                    if CROP_MODE == "inscribed":
                        mask = get_content_mask(img.convert("RGBA"), threshold=BLACK_THRESHOLD)
                        if mask is not None:
                            rect = largest_inscribed_rect(mask, search_max_dim=SEARCH_MAX_DIM)
                            if rect is not None:
                                ImageDraw.Draw(debug_img).rectangle(rect, outline=(255, 0, 0), width=5)
                    else:
                        box = get_content_bbox(img.convert("RGBA"), threshold=BLACK_THRESHOLD)
                        if box is not None:
                            ImageDraw.Draw(debug_img).rectangle(box, outline=(255, 0, 0), width=5)
                    debug_img.save(os.path.join(DEBUG_FOLDER, name_only + "_debug.jpg"))

                cropped = bbox_crop(
                    img,
                    zoom=ZOOM,
                    auto_trim=AUTO_TRIM,
                    threshold=BLACK_THRESHOLD,
                    crop_mode=CROP_MODE,
                    trim_padding=TRIM_PADDING,
                    safety_margin=SAFETY_MARGIN,
                    force_square=FORCE_SQUARE,
                    search_max_dim=SEARCH_MAX_DIM,
                )

                output_filename = name_only + ".png"
                output_path = os.path.join(output_folder, output_filename)

                cropped.save(output_path, "PNG")

        except OSError as e:
            if e.errno == 107:
                failed_files.append((filename, "Koneksi Google Drive terputus"))
                drive_disconnected = True
                break
            failed_files.append((filename, str(e)))
        except Exception as e:
            failed_files.append((filename, str(e)))

    progress.close()

    print(f"\nSelesai. Hasil tersimpan di: {output_folder}")

    if failed_files:
        print(f"\n{len(failed_files)} gambar GAGAL diproses:")
        for filename, reason in failed_files:
            print(f"  - {filename}  ({reason})")
    else:
        print("\nSemua gambar berhasil diproses, tidak ada yang gagal.")

    if drive_disconnected:
        print("\n>> Mount Drive kena disconnect. Jalankan ulang CELL 1,")
        print(">> lalu jalankan CELL ini lagi (file yang sudah OK tidak perlu diulang).")

#APTOS

In [ ]:
INPUT_FOLDER = "/content/drive/MyDrive/Diabetic-Retinopathy/aptos_dataset/train_full/"
OUTPUT_FOLDER = "/content/drive/MyDrive/Diabetic-Retinopathy/Preprocessed image/APTOS/"

In [ ]:
process_folder(INPUT_FOLDER, OUTPUT_FOLDER)

#IDRID


In [ ]:
INPUT_FOLDER = "/content/drive/MyDrive/Diabetic-Retinopathy/idrid_dataset/train_full/"
OUTPUT_FOLDER = "/content/drive/MyDrive/Diabetic-Retinopathy/Preprocessed image/IDRID/"

In [ ]:
process_folder(INPUT_FOLDER, OUTPUT_FOLDER)

#MESSIDOR

In [ ]:
INPUT_FOLDER = "/content/drive/MyDrive/Diabetic-Retinopathy/messidor_dataset/train_full/"
OUTPUT_FOLDER = "/content/drive/MyDrive/Diabetic-Retinopathy/Preprocessed image/MESSIDOR/"

In [ ]:
process_folder(INPUT_FOLDER, OUTPUT_FOLDER)